In [1]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import plotly.graph_objects as go
from src.data.Segment_Slicer import SegmentSlicer 
ss = SegmentSlicer()
from scipy.signal import savgol_filter

In [2]:
raw_parquet = pd.read_parquet('../data/raw/reunion_segments.parquet')
df_parquet = pd.DataFrame(raw_parquet)
df_parquet = df_parquet.rename(columns={'id': 'segment_id'})

In [3]:
ss.cut_segment(df_parquet['altitude_profile'].iloc[0], df_parquet['distance_profile'].iloc[0], df_parquet['coordinates'].iloc[0])

[{'type': 'uphill',
  'category': 'Uncategorized',
  'start_distance': np.float64(13.5),
  'end_distance': np.float64(322.3000000000011),
  'distance': np.float64(308.8000000000011),
  'start_altitude': np.float64(569.5),
  'end_altitude': np.float64(608.5),
  'elevation_gain': np.float64(38.10000000000002),
  'elevation_loss': 0,
  'elevation_change': np.float64(38.10000000000002),
  'grade': np.float64(12.338082901554367),
  'max_grade': np.float64(31.125541125542295),
  'min_grade': np.float64(0.0),
  'grade_variance': np.float64(78.77376579733996),
  'sharp_turns': 0,
  'start_idx': np.int64(5),
  'end_idx': np.int64(96)}]

In [4]:
def plot_elevation_profile(df, segment_id, smooth_window=20):
    
    row = df[df['segment_id'] == segment_id].iloc[0]
    altitude_profile = row['altitude_profile']
    distance_profile = row['distance_profile']
    coordinates = row['coordinates']
    
    segments = ss.cut_segment(altitude_profile, distance_profile, coordinates)
    fig = go.Figure()

    # Smooth the altitude profile if smooth_window is greater than 0
    N = len(altitude_profile)

    # Smooth the altitude profile if smooth_window is greater than 0
    if smooth_window > 0:
        
        # 1. Définir la taille maximale possible pour la fenêtre de lissage.
        #    La taille doit être < N, et nous voulons qu'elle soit impaire.
        
        # Si N est pair, N-1 est impair. Si N est impair, N-2 est impair.
        max_valid_window = N - 1 if N % 2 == 0 else N - 2
        
        # 2. Utiliser la valeur la plus petite entre la fenêtre demandée et la taille maximale valide.
        current_window = min(smooth_window, max_valid_window)
        
        # 3. S'assurer que la fenêtre est impaire (juste au cas où).
        if current_window % 2 == 0:
            current_window -= 1
            
        # 4. S'assurer que la fenêtre est > 1 (le minimum pratique est 3)
        if current_window <= 1:
            # Ne pas lisser si le segment est trop court
            altitude_profile_smoothed = altitude_profile
            print(f"Warning: Segment trop court ({N} points). Lissage ignoré.")
        else:
            # Définir le degré polynomial (doit être < current_window)
            polyorder = 2 if current_window <= 3 else 3
            
            if current_window < polyorder:
                 polyorder = current_window - 1 # Assurance supplémentaire si la fenêtre est très petite
            
            if current_window > 1 : # Répétez la vérification pour la sécurité
                 altitude_profile_smoothed = savgol_filter(altitude_profile, current_window, polyorder)
            else:
                 altitude_profile_smoothed = altitude_profile
                 
    else:
        altitude_profile_smoothed = altitude_profile

    # Calculate grades for hover info
    grades = []
    for i in range(len(altitude_profile_smoothed)):
        if i == 0:
            grades.append(0)
        else:
            dist_diff = distance_profile[i] - distance_profile[i-1]
            elev_diff = altitude_profile_smoothed[i] - altitude_profile_smoothed[i-1]
            grade = (elev_diff / dist_diff * 100) if dist_diff > 0 else 0
            grades.append(grade)

    # Add thick black line for the actual elevation profile with custom hover text
    fig.add_trace(go.Scatter(
        x=distance_profile,
        y=altitude_profile_smoothed,
        mode='lines',
        line=dict(width=2, color='black'),
        name='Elevation Profile',
        hoverinfo='none',  # Disable default hover info
        hovertemplate='<br>'.join([
            '%{customdata[0]}',
            '%{customdata[1]}',
            '%{customdata[2]}'
        ])+'<extra></extra>',
        customdata=np.stack((
            [f"📏 {x:.0f}m" for x in distance_profile], 
            [f"🏔️ {y:.0f}m" for y in altitude_profile_smoothed],
            [f"📊 {g:.1f}%" for g in grades]
        ), axis=-1)
    ))

    min_alt = min(altitude_profile_smoothed)
    baseline = min_alt * 0.9

    # Add colored areas for each segment type below the black line
    for seg in segments:
        if seg['type'] == 'climb':
            color = 'red'
        elif seg['type'] == 'uphill':
            color = 'orange'
        elif seg['type'] == 'flat':
            color = 'blue'
        elif seg['type'] == 'downhill':
            color = 'grey'
        elif seg['type'] == 'descent':
            color = 'green'
        else:
            color = 'blue'

        # Find indices for the segment in the distance_profile
        start_idx = np.searchsorted(distance_profile, seg['start_distance'], side='left')
        end_idx = np.searchsorted(distance_profile, seg['end_distance'], side='right')

        # Create a polygon shape for the filled area
        fig.add_shape(
            type='path',
            path=f'M {distance_profile[start_idx]} {altitude_profile_smoothed[start_idx]} ' +
                 ' '.join([f'L {distance_profile[i]} {altitude_profile_smoothed[i]}' for i in range(start_idx + 1, end_idx)]) +
                 f' L {distance_profile[end_idx-1]} {baseline} ' +
                 ' '.join([f'L {distance_profile[i]} {baseline}' for i in range(end_idx - 2, start_idx - 1, -1)]) +
                 ' Z',
            fillcolor=color,
            opacity=0.5,
            line=dict(width=0)
        )

        # Add annotations for each segment type
        x_mid = (seg['start_distance'] + seg['end_distance']) / 2
        y_mid = (altitude_profile_smoothed[start_idx] + altitude_profile_smoothed[end_idx-1]) / 2
        if seg['type'] == 'climb':
            annotation_text = f"📏 {seg['distance']:.0f}m <br>🏔️ {seg['elevation_change']:.0f}m <br>📈 {seg['grade']:.1f}%"
        elif seg['type'] == 'uphill':
            annotation_text = f"📏 {seg['distance']:.0f}m <br>🏔️ {seg['elevation_change']:.0f}m <br>📈 {seg['grade']:.1f}%"
        elif seg['type'] == 'flat':
            annotation_text = f"📏 {seg['distance']:.0f}m <br>📈 {seg['grade']:.1f}%"
        elif seg['type'] == 'downhill':
            annotation_text = f"📏 {seg['distance']:.0f}m <br>🏔️ {abs(seg['elevation_change']):.0f}m <br>📉 {abs(seg['grade']):.1f}%"
        elif seg['type'] == 'descent':
            annotation_text = f"📏 {seg['distance']:.0f}m <br>🏔️ {abs(seg['elevation_change']):.0f}m <br>📉 {abs(seg['grade']):.1f}%"

        fig.add_annotation(
            x=x_mid,
            y=y_mid + (max(altitude_profile_smoothed) - min(altitude_profile_smoothed)) * 0.1,
            text=annotation_text,
            showarrow=False,
            font=dict(color='black', size=10),
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='black',
            borderwidth=1,
            borderpad=3
        )

    # Update layout
    fig.update_layout(
        title='Trail Elevation Profile with Segment Types',
        xaxis_title='Distance (m)',
        yaxis_title='Altitude (m)',
        yaxis=dict(
            range=[min_alt * 0.9, max(altitude_profile_smoothed) * 1.1],
            showgrid=False
        ),
        xaxis=dict(showgrid=False),
        showlegend=False,
        hovermode='x',
        margin=dict(t=40),
        height=600
    )
    fig.show()

In [5]:
plot_elevation_profile(df_parquet, 4905002, smooth_window=20)

In [6]:
plot_elevation_profile(df_parquet, 8388863, smooth_window=20)